In [3]:
from __future__ import annotations

import os
import csv
import json
import time
import math
import random
import shutil
from pathlib import Path
from typing import Dict, Any, List, Optional, Tuple

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm


# ============================================================
# ROOTS
# ============================================================

ROOT = Path.cwd()

PT_DIR = ROOT / "generated_pt"
RUN_DIR = ROOT / "runs_fractional_scenarios"

SUMMARY_CSV = ROOT / "summary_fractional_scenarios.csv"
SUMMARY_JSON = ROOT / "summary_fractional_scenarios.json"

RUN_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# GLOBAL CONFIG
# ============================================================

SEED = 42

RUNS = 3
EPOCHS = 10
VAL_SPLIT = 0.2
BATCH_SIZE = 64
NUM_WORKERS = 0

RESUME = True
RERUN_INCOMPLETE = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("ROOT:", ROOT)
print("PT_DIR:", PT_DIR)
print("RUN_DIR:", RUN_DIR)
print("DEVICE:", DEVICE)


# ============================================================
# DATASET CONFIGS — SAME AS BEFORE
# ============================================================

DATASET_CONFIGS = [
    {
        "dataset_name": "small_D2_L64",
        "n_samples": 5000,
        "depth": 2,
        "max_len": 64,
        "seed_offset": 0,
    },
    {
        "dataset_name": "base_D3_L128",
        "n_samples": 10000,
        "depth": 3,
        "max_len": 128,
        "seed_offset": 1000,
    },
    {
        "dataset_name": "hard_D4_L160",
        "n_samples": 12000,
        "depth": 4,
        "max_len": 160,
        "seed_offset": 2000,
    },
]


# ============================================================
# TRANSFORMER CONFIGS — SAME AS BEFORE
# ============================================================

MODEL_CONFIGS = [
    {
        "model_name": "T_small_mean",
        "d_model": 96,
        "nhead": 4,
        "num_layers": 2,
        "dim_feedforward": 256,
        "dropout": 0.10,
        "pooling": "mean",
    },
    {
        "model_name": "T_base_mean",
        "d_model": 128,
        "nhead": 4,
        "num_layers": 3,
        "dim_feedforward": 512,
        "dropout": 0.10,
        "pooling": "mean",
    },
    {
        "model_name": "T_base_cls",
        "d_model": 128,
        "nhead": 4,
        "num_layers": 3,
        "dim_feedforward": 512,
        "dropout": 0.10,
        "pooling": "cls",
    },
    {
        "model_name": "T_deep_mean",
        "d_model": 128,
        "nhead": 4,
        "num_layers": 5,
        "dim_feedforward": 512,
        "dropout": 0.15,
        "pooling": "mean",
    },
    {
        "model_name": "T_wide_mean",
        "d_model": 192,
        "nhead": 6,
        "num_layers": 3,
        "dim_feedforward": 768,
        "dropout": 0.10,
        "pooling": "mean",
    },
]


# ============================================================
# FRACTIONAL SCENARIOS
# ============================================================
# target:
#   all         — all parameters
#   head        — classifier only
#   ffn         — Transformer feed-forward layers only
#   attention   — self-attention only
#   embeddings  — token + position embeddings only
#
# mode:
#   none        — ordinary AdamW
#   replace     — replace gradient with fractional memory
#   mix         — g = (1-lambda) * g_original + lambda * g_fractional
#
# start_epoch:
#   1 means from the beginning
#   4 means warmup for epochs 1-3, fractional from epoch 4

SCENARIO_CONFIGS = [
    {
        "scenario_name": "baseline_adamw",
        "base_optimizer": "adamw",
        "lr": 3e-4,
        "weight_decay": 1e-2,
        "mode": "none",
        "target": "none",
        "alpha": None,
        "beta": None,
        "mix_lambda": 0.0,
        "start_epoch": None,
    },

    # Naive full replacement, close to previous experiment.
    {
        "scenario_name": "naive_all_replace_a08",
        "base_optimizer": "adamw",
        "lr": 3e-4,
        "weight_decay": 1e-2,
        "mode": "replace",
        "target": "all",
        "alpha": 0.80,
        "beta": 0.90,
        "mix_lambda": 1.0,
        "start_epoch": 1,
    },

    # A: delayed fractional memory.
    {
        "scenario_name": "delayed_all_replace_a08_warm3",
        "base_optimizer": "adamw",
        "lr": 3e-4,
        "weight_decay": 1e-2,
        "mode": "replace",
        "target": "all",
        "alpha": 0.80,
        "beta": 0.90,
        "mix_lambda": 1.0,
        "start_epoch": 4,
    },

    # C: mixed fractional memory.
    {
        "scenario_name": "mixed_all_a08_lam025",
        "base_optimizer": "adamw",
        "lr": 3e-4,
        "weight_decay": 1e-2,
        "mode": "mix",
        "target": "all",
        "alpha": 0.80,
        "beta": 0.90,
        "mix_lambda": 0.25,
        "start_epoch": 1,
    },
    {
        "scenario_name": "mixed_all_a08_lam050",
        "base_optimizer": "adamw",
        "lr": 3e-4,
        "weight_decay": 1e-2,
        "mode": "mix",
        "target": "all",
        "alpha": 0.80,
        "beta": 0.90,
        "mix_lambda": 0.50,
        "start_epoch": 1,
    },

    # B: selective fractional memory.
    {
        "scenario_name": "head_only_replace_a08",
        "base_optimizer": "adamw",
        "lr": 3e-4,
        "weight_decay": 1e-2,
        "mode": "replace",
        "target": "head",
        "alpha": 0.80,
        "beta": 0.90,
        "mix_lambda": 1.0,
        "start_epoch": 1,
    },
    {
        "scenario_name": "ffn_only_replace_a08",
        "base_optimizer": "adamw",
        "lr": 3e-4,
        "weight_decay": 1e-2,
        "mode": "replace",
        "target": "ffn",
        "alpha": 0.80,
        "beta": 0.90,
        "mix_lambda": 1.0,
        "start_epoch": 1,
    },
    {
        "scenario_name": "attention_only_replace_a08",
        "base_optimizer": "adamw",
        "lr": 3e-4,
        "weight_decay": 1e-2,
        "mode": "replace",
        "target": "attention",
        "alpha": 0.80,
        "beta": 0.90,
        "mix_lambda": 1.0,
        "start_epoch": 1,
    },
    {
        "scenario_name": "embeddings_only_replace_a08",
        "base_optimizer": "adamw",
        "lr": 3e-4,
        "weight_decay": 1e-2,
        "mode": "replace",
        "target": "embeddings",
        "alpha": 0.80,
        "beta": 0.90,
        "mix_lambda": 1.0,
        "start_epoch": 1,
    },

    # Combined: delayed + mixed.
    {
        "scenario_name": "delayed_mixed_all_a08_lam025_warm3",
        "base_optimizer": "adamw",
        "lr": 3e-4,
        "weight_decay": 1e-2,
        "mode": "mix",
        "target": "all",
        "alpha": 0.80,
        "beta": 0.90,
        "mix_lambda": 0.25,
        "start_epoch": 4,
    },
    {
        "scenario_name": "delayed_mixed_head_a08_lam025_warm3",
        "base_optimizer": "adamw",
        "lr": 3e-4,
        "weight_decay": 1e-2,
        "mode": "mix",
        "target": "head",
        "alpha": 0.80,
        "beta": 0.90,
        "mix_lambda": 0.25,
        "start_epoch": 4,
    },
]


# ============================================================
# VOCABULARY
# ============================================================

VOCAB = ["[PAD]", "[CLS]", "[", "]", "MAX", "MIN", "SUM", "PROD", "MED"] + [
    str(i) for i in range(10)
]

VOCAB_DICT = {tok: i for i, tok in enumerate(VOCAB)}
PAD_ID = VOCAB_DICT["[PAD]"]


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


def safe_name(x: Any) -> str:
    return (
        str(x)
        .replace(" ", "_")
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
        .replace(".", "p")
        .replace("'", "")
        .replace('"', "")
    )


def log(msg: str, path: Path) -> None:
    print(msg)
    with open(path, "a", encoding="utf-8") as f:
        f.write(msg + "\n")


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def expected_pt_path(cfg: Dict[str, Any]) -> Path:
    dataset_seed = SEED + cfg["seed_offset"]

    return PT_DIR / (
        f"listops_{safe_name(cfg['dataset_name'])}"
        f"_N{cfg['n_samples']}_D{cfg['depth']}_L{cfg['max_len']}_seed{dataset_seed}.pt"
    )


def find_dataset_pt(cfg: Dict[str, Any]) -> Path:
    expected = expected_pt_path(cfg)

    if expected.exists():
        return expected

    pattern = f"listops_{safe_name(cfg['dataset_name'])}_*.pt"
    matches = sorted(PT_DIR.glob(pattern))

    if len(matches) == 0:
        raise FileNotFoundError(
            f"No .pt file found for dataset {cfg['dataset_name']} in {PT_DIR}. "
            f"Expected: {expected}"
        )

    print(f"Expected file not found, using matched file: {matches[0]}")
    return matches[0]


def get_exp_dir(
    dataset_name: str,
    model_name: str,
    scenario_name: str,
    run_id: int,
) -> Path:
    return (
        RUN_DIR
        / safe_name(dataset_name)
        / safe_name(model_name)
        / safe_name(scenario_name)
        / f"run_{run_id}"
    )


def is_run_complete(exp_dir: Path, expected_epochs: int) -> bool:
    history_path = exp_dir / "history.json"
    best_model_path = exp_dir / "best_model.pt"
    final_model_path = exp_dir / "final_model.pt"

    if not history_path.exists():
        return False
    if not best_model_path.exists():
        return False
    if not final_model_path.exists():
        return False

    try:
        with open(history_path, "r", encoding="utf-8") as f:
            history = json.load(f)

        vals = history.get("val_macro_f1", None)

        if not isinstance(vals, list):
            return False

        if len(vals) < expected_epochs:
            return False

    except Exception:
        return False

    return True


def remove_incomplete_run(exp_dir: Path) -> None:
    if exp_dir.exists():
        print(f"Removing incomplete run: {exp_dir}")
        shutil.rmtree(exp_dir)


# ============================================================
# DATASET
# ============================================================

class GeneratedListOpsDataset(Dataset):
    def __init__(self, pt_path: Path):
        data = torch.load(pt_path, map_location="cpu")

        self.X = data["X"].long()
        self.Y = data["Y"].long()

        if self.Y.ndim != 1:
            self.Y = self.Y.view(-1)

        self.dataset_config = data.get("dataset_config", {})
        self.lengths = data.get("lengths", None)

    def __len__(self) -> int:
        return len(self.X)

    def __getitem__(self, idx: int):
        return self.X[idx], self.Y[idx]


def dataset_info_from_pt(pt_path: Path) -> Dict[str, Any]:
    data = torch.load(pt_path, map_location="cpu")

    X = data["X"]
    Y = data["Y"]

    info = {
        "pt_path": str(pt_path),
        "dataset_name": data["dataset_config"]["dataset_name"],
        "n_samples": int(X.shape[0]),
        "seq_len": int(X.shape[1]),
        "vocab_size": int(X.max().item()) + 1,
        "num_classes": int(Y.max().item()) + 1,
        "unique_classes": sorted([int(v) for v in Y.unique().tolist()]),
        "dataset_config": data.get("dataset_config", {}),
        "seed": data.get("seed", None),
    }

    if "lengths" in data:
        lengths = data["lengths"]
        info["mean_length"] = float(lengths.float().mean().item())
        info["min_length"] = int(lengths.min().item())
        info["max_length"] = int(lengths.max().item())

    return info


# ============================================================
# MODEL
# ============================================================

class GeneratedTransformerClassifier(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        num_classes: int,
        seq_len: int,
        d_model: int,
        nhead: int,
        num_layers: int,
        dim_feedforward: int,
        dropout: float,
        pooling: str,
        pad_idx: int = 0,
    ):
        super().__init__()

        if d_model % nhead != 0:
            raise ValueError(f"d_model={d_model} must be divisible by nhead={nhead}")

        if pooling not in {"mean", "cls"}:
            raise ValueError("pooling must be 'mean' or 'cls'")

        self.pad_idx = pad_idx
        self.pooling = pooling

        self.token_embed = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=d_model,
            padding_idx=pad_idx,
        )

        self.pos_embed = nn.Embedding(
            num_embeddings=seq_len,
            embedding_dim=d_model,
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer=encoder_layer,
            num_layers=num_layers,
        )

        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len = x.shape

        pos = torch.arange(seq_len, device=x.device)
        pos = pos.unsqueeze(0).expand(batch_size, seq_len)

        pad_mask = x.eq(self.pad_idx)

        h = self.token_embed(x) + self.pos_embed(pos)
        h = self.encoder(h, src_key_padding_mask=pad_mask)

        if self.pooling == "cls":
            pooled = h[:, 0, :]
        else:
            mask = (~pad_mask).float().unsqueeze(-1)
            h = h * mask
            denom = mask.sum(dim=1).clamp(min=1.0)
            pooled = h.sum(dim=1) / denom

        pooled = self.norm(pooled)
        pooled = self.dropout(pooled)

        return self.classifier(pooled)


# ============================================================
# FRACTIONAL CONTROLLER
# ============================================================

class FractionalGradientController:
    """
    Supports:
      1. replace mode:
            grad <- fractional_memory
      2. mix mode:
            grad <- (1-lambda) * grad_original + lambda * fractional_memory
      3. delayed start:
            applied only from start_epoch
      4. selective target:
            all / head / ffn / attention / embeddings
    """

    def __init__(
        self,
        model: nn.Module,
        mode: str,
        target: str,
        alpha: Optional[float],
        beta: Optional[float],
        mix_lambda: float = 1.0,
        start_epoch: Optional[int] = 1,
    ):
        self.model = model
        self.mode = mode
        self.target = target
        self.alpha = alpha
        self.beta = beta
        self.mix_lambda = mix_lambda
        self.start_epoch = start_epoch

        self.enabled = mode in {"replace", "mix"}

        if not self.enabled:
            self.coeff = None
            self.memory = {}
            return

        if alpha is None or beta is None:
            raise ValueError("alpha and beta must be provided for fractional modes")

        if not (0.0 < alpha <= 1.0):
            raise ValueError("alpha must be in (0, 1]")

        if not (0.0 <= beta < 1.0):
            raise ValueError("beta must be in [0, 1)")

        if not (0.0 <= mix_lambda <= 1.0):
            raise ValueError("mix_lambda must be in [0, 1]")

        if mode not in {"replace", "mix"}:
            raise ValueError("mode must be one of: none, replace, mix")

        self.coeff = 1.0 / math.gamma(2.0 - alpha)
        self.memory: Dict[str, torch.Tensor] = {}

    def _matches_target(self, name: str) -> bool:
        if self.target == "all":
            return True

        if self.target == "head":
            return name.startswith("classifier")

        if self.target == "embeddings":
            return name.startswith("token_embed") or name.startswith("pos_embed")

        if self.target == "attention":
            return "self_attn" in name

        if self.target == "ffn":
            return "linear1" in name or "linear2" in name

        if self.target == "none":
            return False

        raise ValueError(f"Unknown target: {self.target}")

    def apply(self, current_epoch: int) -> int:
        if not self.enabled:
            return 0

        if self.start_epoch is not None and current_epoch < self.start_epoch:
            return 0

        applied = 0

        with torch.no_grad():
            for name, p in self.model.named_parameters():
                if p.grad is None:
                    continue

                if not self._matches_target(name):
                    continue

                g_original = p.grad.detach()

                if name not in self.memory:
                    self.memory[name] = torch.zeros_like(g_original)

                mem = self.memory[name]
                mem.mul_(self.beta)
                mem.add_(g_original, alpha=(1.0 - self.beta) * self.coeff)

                if self.mode == "replace":
                    p.grad.copy_(mem)

                elif self.mode == "mix":
                    mixed = (1.0 - self.mix_lambda) * g_original + self.mix_lambda * mem
                    p.grad.copy_(mixed)

                else:
                    raise ValueError(f"Unknown mode: {self.mode}")

                applied += 1

        return applied


# ============================================================
# OPTIMIZER
# ============================================================

def make_optimizer(model: nn.Module, cfg: Dict[str, Any]):
    base_name = cfg.get("base_optimizer", "adamw")
    lr = cfg.get("lr", 3e-4)
    weight_decay = cfg.get("weight_decay", 1e-2)

    if base_name == "adamw":
        return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    if base_name == "adam":
        return torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    if base_name == "sgd":
        return torch.optim.SGD(
            model.parameters(),
            lr=lr,
            momentum=cfg.get("momentum", 0.9),
            weight_decay=weight_decay,
        )

    raise ValueError(f"Unknown base optimizer: {base_name}")


# ============================================================
# METRICS
# ============================================================

def classification_metrics(
    preds: torch.Tensor,
    targets: torch.Tensor,
    num_classes: int,
) -> Dict[str, float]:
    preds = preds.detach().cpu()
    targets = targets.detach().cpu()

    accuracy = (preds == targets).float().mean().item()

    macro_precision = []
    macro_recall = []
    macro_f1 = []
    weighted_f1 = []

    total_support = len(targets)

    global_tp = 0
    global_fp = 0
    global_fn = 0

    for c in range(num_classes):
        tp = ((preds == c) & (targets == c)).sum().item()
        fp = ((preds == c) & (targets != c)).sum().item()
        fn = ((preds != c) & (targets == c)).sum().item()
        support = (targets == c).sum().item()

        precision = tp / (tp + fp + 1e-12)
        recall = tp / (tp + fn + 1e-12)
        f1 = 2.0 * precision * recall / (precision + recall + 1e-12)

        macro_precision.append(precision)
        macro_recall.append(recall)
        macro_f1.append(f1)
        weighted_f1.append(f1 * support)

        global_tp += tp
        global_fp += fp
        global_fn += fn

    micro_precision = global_tp / (global_tp + global_fp + 1e-12)
    micro_recall = global_tp / (global_tp + global_fn + 1e-12)
    micro_f1 = 2.0 * micro_precision * micro_recall / (
        micro_precision + micro_recall + 1e-12
    )

    return {
        "accuracy": accuracy,
        "macro_precision": sum(macro_precision) / num_classes,
        "macro_recall": sum(macro_recall) / num_classes,
        "macro_f1": sum(macro_f1) / num_classes,
        "micro_f1": micro_f1,
        "weighted_f1": sum(weighted_f1) / max(total_support, 1),
    }


def compute_grad_norm(model: nn.Module) -> float:
    total = 0.0

    for p in model.parameters():
        if p.grad is not None:
            norm = p.grad.detach().data.norm(2).item()
            total += norm ** 2

    return total ** 0.5


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    num_classes: int,
) -> Dict[str, float]:
    model.eval()

    total_loss = 0.0
    total_confidence = 0.0
    total_items = 0

    all_preds = []
    all_targets = []

    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        logits = model(x)
        loss = criterion(logits, y)

        probs = torch.softmax(logits, dim=1)
        confidence = probs.max(dim=1)[0]
        preds = logits.argmax(dim=1)

        batch_size = y.size(0)

        total_loss += loss.item() * batch_size
        total_confidence += confidence.sum().item()
        total_items += batch_size

        all_preds.append(preds)
        all_targets.append(y)

    all_preds = torch.cat(all_preds)
    all_targets = torch.cat(all_targets)

    cls = classification_metrics(all_preds, all_targets, num_classes)

    return {
        "loss": total_loss / total_items,
        "confidence": total_confidence / total_items,
        **cls,
    }


# ============================================================
# TRAIN ONE
# ============================================================

def train_one(
    pt_path: Path,
    dataset_info: Dict[str, Any],
    model_cfg: Dict[str, Any],
    scenario_cfg: Dict[str, Any],
    run_id: int,
    seed: int,
) -> Dict[str, Any]:

    set_seed(seed)

    dataset_name = dataset_info["dataset_name"]
    model_name = model_cfg["model_name"]
    scenario_name = scenario_cfg["scenario_name"]

    exp_dir = get_exp_dir(dataset_name, model_name, scenario_name, run_id)

    if RESUME and is_run_complete(exp_dir, EPOCHS):
        print(f"SKIP complete run: {exp_dir}")
        return {
            "status": "skipped_complete",
            "dataset_name": dataset_name,
            "model_name": model_name,
            "scenario_name": scenario_name,
            "run_id": run_id,
            "exp_dir": str(exp_dir),
        }

    if RESUME and exp_dir.exists() and not is_run_complete(exp_dir, EPOCHS):
        if RERUN_INCOMPLETE:
            remove_incomplete_run(exp_dir)
        else:
            print(f"SKIP incomplete run: {exp_dir}")
            return {
                "status": "skipped_incomplete",
                "dataset_name": dataset_name,
                "model_name": model_name,
                "scenario_name": scenario_name,
                "run_id": run_id,
                "exp_dir": str(exp_dir),
            }

    exp_dir.mkdir(parents=True, exist_ok=True)

    log_path = exp_dir / "log.txt"
    history_path = exp_dir / "history.json"

    dataset_info_path = exp_dir / "dataset_info.json"
    model_config_path = exp_dir / "model_config.json"
    scenario_config_path = exp_dir / "scenario_config.json"

    initial_model_path = exp_dir / "initial_model.pt"
    best_model_path = exp_dir / "best_model.pt"
    final_model_path = exp_dir / "final_model.pt"

    with open(dataset_info_path, "w", encoding="utf-8") as f:
        json.dump(dataset_info, f, indent=4, ensure_ascii=False)

    with open(model_config_path, "w", encoding="utf-8") as f:
        json.dump(model_cfg, f, indent=4, ensure_ascii=False)

    with open(scenario_config_path, "w", encoding="utf-8") as f:
        json.dump(scenario_cfg, f, indent=4, ensure_ascii=False)

    dataset = GeneratedListOpsDataset(pt_path)

    val_size = int(len(dataset) * VAL_SPLIT)
    train_size = len(dataset) - val_size

    split_generator = torch.Generator().manual_seed(seed)

    train_dataset, val_dataset = random_split(
        dataset,
        [train_size, val_size],
        generator=split_generator,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    model = GeneratedTransformerClassifier(
        vocab_size=dataset_info["vocab_size"],
        num_classes=dataset_info["num_classes"],
        seq_len=dataset_info["seq_len"],
        d_model=model_cfg["d_model"],
        nhead=model_cfg["nhead"],
        num_layers=model_cfg["num_layers"],
        dim_feedforward=model_cfg["dim_feedforward"],
        dropout=model_cfg["dropout"],
        pooling=model_cfg["pooling"],
        pad_idx=PAD_ID,
    ).to(DEVICE)

    num_params = count_parameters(model)

    torch.save(
        {
            "state_dict": model.state_dict(),
            "dataset_info": dataset_info,
            "model_config": model_cfg,
            "scenario_config": scenario_cfg,
            "num_params": num_params,
            "seed": seed,
        },
        initial_model_path,
    )

    optimizer = make_optimizer(model, scenario_cfg)
    criterion = nn.CrossEntropyLoss()

    frac_controller = FractionalGradientController(
        model=model,
        mode=scenario_cfg["mode"],
        target=scenario_cfg["target"],
        alpha=scenario_cfg["alpha"],
        beta=scenario_cfg["beta"],
        mix_lambda=scenario_cfg["mix_lambda"],
        start_epoch=scenario_cfg["start_epoch"],
    )

    history = {
        "train_loss": [],
        "train_accuracy": [],
        "train_macro_precision": [],
        "train_macro_recall": [],
        "train_macro_f1": [],
        "train_micro_f1": [],
        "train_weighted_f1": [],
        "train_confidence": [],

        "val_loss": [],
        "val_accuracy": [],
        "val_macro_precision": [],
        "val_macro_recall": [],
        "val_macro_f1": [],
        "val_micro_f1": [],
        "val_weighted_f1": [],
        "val_confidence": [],

        "grad_norm": [],
        "fractional_applied_params": [],
        "epoch_time": [],
        "throughput_samples_per_sec": [],
        "lr": [],
    }

    best_val_macro_f1 = -1.0
    best_val_accuracy = -1.0
    best_val_loss = float("inf")
    best_epoch = -1

    run_start = time.time()

    log("=" * 100, log_path)
    log(f"DATASET      : {dataset_name}", log_path)
    log(f"MODEL        : {model_name}", log_path)
    log(f"SCENARIO     : {scenario_name}", log_path)
    log(f"RUN          : {run_id}", log_path)
    log(f"SEED         : {seed}", log_path)
    log(f"DEVICE       : {DEVICE}", log_path)
    log(f"PARAMETERS   : {num_params:,}", log_path)
    log(f"TRAIN SIZE   : {train_size}", log_path)
    log(f"VAL SIZE     : {val_size}", log_path)
    log(f"SCENARIO CFG : {scenario_cfg}", log_path)
    log("=" * 100, log_path)

    for epoch in range(1, EPOCHS + 1):
        epoch_start = time.time()

        model.train()

        total_loss = 0.0
        total_confidence = 0.0
        total_items = 0

        all_preds = []
        all_targets = []

        grad_norm_sum = 0.0
        grad_steps = 0
        fractional_applied_sum = 0

        for x, y in train_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)

            logits = model(x)
            loss = criterion(logits, y)

            loss.backward()

            applied = frac_controller.apply(current_epoch=epoch)
            fractional_applied_sum += applied

            grad_norm = compute_grad_norm(model)
            grad_norm_sum += grad_norm
            grad_steps += 1

            optimizer.step()

            with torch.no_grad():
                probs = torch.softmax(logits, dim=1)
                confidence = probs.max(dim=1)[0]
                preds = logits.argmax(dim=1)

            batch_size = y.size(0)

            total_loss += loss.item() * batch_size
            total_confidence += confidence.sum().item()
            total_items += batch_size

            all_preds.append(preds.detach())
            all_targets.append(y.detach())

        train_preds = torch.cat(all_preds)
        train_targets = torch.cat(all_targets)

        train_cls = classification_metrics(
            train_preds,
            train_targets,
            num_classes=dataset_info["num_classes"],
        )

        train_metrics = {
            "loss": total_loss / total_items,
            "confidence": total_confidence / total_items,
            **train_cls,
        }

        val_metrics = evaluate(
            model=model,
            loader=val_loader,
            criterion=criterion,
            num_classes=dataset_info["num_classes"],
        )

        epoch_time = time.time() - epoch_start
        throughput = total_items / max(epoch_time, 1e-12)
        avg_grad_norm = grad_norm_sum / max(grad_steps, 1)
        avg_fractional_applied = fractional_applied_sum / max(grad_steps, 1)
        current_lr = optimizer.param_groups[0]["lr"]

        history["train_loss"].append(train_metrics["loss"])
        history["train_accuracy"].append(train_metrics["accuracy"])
        history["train_macro_precision"].append(train_metrics["macro_precision"])
        history["train_macro_recall"].append(train_metrics["macro_recall"])
        history["train_macro_f1"].append(train_metrics["macro_f1"])
        history["train_micro_f1"].append(train_metrics["micro_f1"])
        history["train_weighted_f1"].append(train_metrics["weighted_f1"])
        history["train_confidence"].append(train_metrics["confidence"])

        history["val_loss"].append(val_metrics["loss"])
        history["val_accuracy"].append(val_metrics["accuracy"])
        history["val_macro_precision"].append(val_metrics["macro_precision"])
        history["val_macro_recall"].append(val_metrics["macro_recall"])
        history["val_macro_f1"].append(val_metrics["macro_f1"])
        history["val_micro_f1"].append(val_metrics["micro_f1"])
        history["val_weighted_f1"].append(val_metrics["weighted_f1"])
        history["val_confidence"].append(val_metrics["confidence"])

        history["grad_norm"].append(avg_grad_norm)
        history["fractional_applied_params"].append(avg_fractional_applied)
        history["epoch_time"].append(epoch_time)
        history["throughput_samples_per_sec"].append(throughput)
        history["lr"].append(current_lr)

        if val_metrics["macro_f1"] > best_val_macro_f1:
            best_val_macro_f1 = val_metrics["macro_f1"]
            best_val_accuracy = val_metrics["accuracy"]
            best_val_loss = val_metrics["loss"]
            best_epoch = epoch

            torch.save(
                {
                    "state_dict": model.state_dict(),
                    "dataset_info": dataset_info,
                    "model_config": model_cfg,
                    "scenario_config": scenario_cfg,
                    "epoch": best_epoch,
                    "best_val_macro_f1": best_val_macro_f1,
                    "best_val_accuracy": best_val_accuracy,
                    "best_val_loss": best_val_loss,
                    "num_params": num_params,
                    "seed": seed,
                },
                best_model_path,
            )

        log(
            f"Epoch {epoch:03d}/{EPOCHS} | "
            f"Train Loss {train_metrics['loss']:.4f} | "
            f"Train Acc {train_metrics['accuracy']:.4f} | "
            f"Train MacroF1 {train_metrics['macro_f1']:.4f} | "
            f"Val Loss {val_metrics['loss']:.4f} | "
            f"Val Acc {val_metrics['accuracy']:.4f} | "
            f"Val MacroF1 {val_metrics['macro_f1']:.4f} | "
            f"Val WeightedF1 {val_metrics['weighted_f1']:.4f} | "
            f"Val Conf {val_metrics['confidence']:.4f} | "
            f"Grad {avg_grad_norm:.4f} | "
            f"FracApplied {avg_fractional_applied:.1f} | "
            f"Throughput {throughput:.1f} samp/s | "
            f"Time {epoch_time:.2f}s",
            log_path,
        )

    total_time = time.time() - run_start

    torch.save(
        {
            "state_dict": model.state_dict(),
            "dataset_info": dataset_info,
            "model_config": model_cfg,
            "scenario_config": scenario_cfg,
            "final_epoch": EPOCHS,
            "num_params": num_params,
            "seed": seed,
        },
        final_model_path,
    )

    with open(history_path, "w", encoding="utf-8") as f:
        json.dump(history, f, indent=4, ensure_ascii=False)

    log("-" * 100, log_path)
    log(f"BEST EPOCH          : {best_epoch}", log_path)
    log(f"BEST VAL MACRO F1   : {best_val_macro_f1:.6f}", log_path)
    log(f"BEST VAL ACC        : {best_val_accuracy:.6f}", log_path)
    log(f"BEST VAL LOSS       : {best_val_loss:.6f}", log_path)
    log(f"TOTAL RUN TIME      : {total_time:.2f} sec", log_path)
    log("-" * 100, log_path)

    result = {
        "status": "trained",

        "dataset_name": dataset_name,
        "pt_path": str(pt_path),
        "n_samples": dataset_info["n_samples"],
        "seq_len": dataset_info["seq_len"],
        "vocab_size": dataset_info["vocab_size"],
        "num_classes": dataset_info["num_classes"],
        "mean_length": dataset_info.get("mean_length"),
        "max_length": dataset_info.get("max_length"),

        "model_name": model_name,
        "d_model": model_cfg["d_model"],
        "nhead": model_cfg["nhead"],
        "num_layers": model_cfg["num_layers"],
        "dim_feedforward": model_cfg["dim_feedforward"],
        "dropout": model_cfg["dropout"],
        "pooling": model_cfg["pooling"],

        "scenario_name": scenario_name,
        "base_optimizer": scenario_cfg.get("base_optimizer"),
        "lr": scenario_cfg.get("lr"),
        "weight_decay": scenario_cfg.get("weight_decay"),
        "mode": scenario_cfg.get("mode"),
        "target": scenario_cfg.get("target"),
        "alpha": scenario_cfg.get("alpha"),
        "beta": scenario_cfg.get("beta"),
        "mix_lambda": scenario_cfg.get("mix_lambda"),
        "start_epoch": scenario_cfg.get("start_epoch"),

        "run_id": run_id,
        "seed": seed,
        "num_params": num_params,

        "best_epoch": best_epoch,
        "best_val_macro_f1": best_val_macro_f1,
        "best_val_accuracy": best_val_accuracy,
        "best_val_loss": best_val_loss,

        "final_train_loss": history["train_loss"][-1],
        "final_train_accuracy": history["train_accuracy"][-1],
        "final_train_macro_f1": history["train_macro_f1"][-1],

        "final_val_loss": history["val_loss"][-1],
        "final_val_accuracy": history["val_accuracy"][-1],
        "final_val_macro_f1": history["val_macro_f1"][-1],
        "final_val_micro_f1": history["val_micro_f1"][-1],
        "final_val_weighted_f1": history["val_weighted_f1"][-1],
        "final_val_confidence": history["val_confidence"][-1],

        "mean_grad_norm": sum(history["grad_norm"]) / len(history["grad_norm"]),
        "mean_fractional_applied_params": (
            sum(history["fractional_applied_params"])
            / len(history["fractional_applied_params"])
        ),
        "mean_epoch_time": sum(history["epoch_time"]) / len(history["epoch_time"]),
        "mean_throughput": (
            sum(history["throughput_samples_per_sec"])
            / len(history["throughput_samples_per_sec"])
        ),

        "total_time_sec": total_time,

        "exp_dir": str(exp_dir),
        "initial_model_path": str(initial_model_path),
        "best_model_path": str(best_model_path),
        "final_model_path": str(final_model_path),
        "history_path": str(history_path),
        "log_path": str(log_path),
    }

    return result


# ============================================================
# SUMMARY
# ============================================================

def save_summary(results: List[Dict[str, Any]]) -> None:
    if not results:
        return

    with open(SUMMARY_JSON, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=4, ensure_ascii=False)

    fieldnames = sorted(set().union(*(r.keys() for r in results)))

    with open(SUMMARY_CSV, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        for row in results:
            writer.writerow(row)

    print("\nSummary updated:")
    print(" ", SUMMARY_JSON)
    print(" ", SUMMARY_CSV)


# ============================================================
# MAIN
# ============================================================

def main() -> None:
    set_seed(SEED)

    print("=" * 100)
    print("FRACTIONAL SCENARIOS EXPERIMENT")
    print("Same datasets + same Transformers; new fractional mechanisms")
    print("=" * 100)

    pt_paths = []

    for cfg in DATASET_CONFIGS:
        pt_path = find_dataset_pt(cfg)
        pt_paths.append(pt_path)

    total_jobs = len(pt_paths) * len(MODEL_CONFIGS) * len(SCENARIO_CONFIGS) * RUNS

    print("Datasets:", len(pt_paths))
    print("Models:", len(MODEL_CONFIGS))
    print("Scenarios:", len(SCENARIO_CONFIGS))
    print("Runs:", RUNS)
    print("Total jobs:", total_jobs)

    all_results = []

    job_id = 0
    trained_count = 0
    skipped_complete = 0
    skipped_incomplete = 0

    for pt_path in pt_paths:
        dataset_info = dataset_info_from_pt(pt_path)

        for model_cfg in MODEL_CONFIGS:
            for scenario_cfg in SCENARIO_CONFIGS:
                for run_id in range(RUNS):
                    job_id += 1
                    run_seed = SEED + job_id

                    print("\n" + "#" * 100)
                    print(f"JOB {job_id}/{total_jobs}")
                    print(f"DATASET  : {dataset_info['dataset_name']}")
                    print(f"MODEL    : {model_cfg['model_name']}")
                    print(f"SCENARIO : {scenario_cfg['scenario_name']}")
                    print(f"RUN      : {run_id}")
                    print(f"SEED     : {run_seed}")
                    print("#" * 100)

                    result = train_one(
                        pt_path=pt_path,
                        dataset_info=dataset_info,
                        model_cfg=model_cfg,
                        scenario_cfg=scenario_cfg,
                        run_id=run_id,
                        seed=run_seed,
                    )

                    status = result.get("status")

                    if status == "skipped_complete":
                        skipped_complete += 1
                        print("Already complete, not added to new summary.")
                    elif status == "skipped_incomplete":
                        skipped_incomplete += 1
                        all_results.append(result)
                        save_summary(all_results)
                    else:
                        trained_count += 1
                        all_results.append(result)
                        save_summary(all_results)

    print("\nALL DONE")
    print("Run dir:", RUN_DIR)
    print("Summary CSV:", SUMMARY_CSV)
    print("Summary JSON:", SUMMARY_JSON)

    print("\nCOUNTS")
    print("Total jobs:", total_jobs)
    print("Trained now:", trained_count)
    print("Skipped complete:", skipped_complete)
    print("Skipped incomplete:", skipped_incomplete)


if __name__ == "__main__":
    main()

ROOT: /home/tahiti/Malashin_Projects
PT_DIR: /home/tahiti/Malashin_Projects/generated_pt
RUN_DIR: /home/tahiti/Malashin_Projects/runs_fractional_scenarios
DEVICE: cpu
FRACTIONAL SCENARIOS EXPERIMENT
Same datasets + same Transformers; new fractional mechanisms
Datasets: 3
Models: 5
Scenarios: 11
Runs: 3
Total jobs: 495

####################################################################################################
JOB 1/495
DATASET  : small_D2_L64
MODEL    : T_small_mean
SCENARIO : baseline_adamw
RUN      : 0
SEED     : 43
####################################################################################################


/tmp/ipykernel_152190/478453138.py:531: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(


DATASET      : small_D2_L64
MODEL        : T_small_mean
SCENARIO     : baseline_adamw
RUN          : 0
SEED         : 43
DEVICE       : cpu
PARAMETERS   : 183,402
TRAIN SIZE   : 4000
VAL SIZE     : 1000
SCENARIO CFG : {'scenario_name': 'baseline_adamw', 'base_optimizer': 'adamw', 'lr': 0.0003, 'weight_decay': 0.01, 'mode': 'none', 'target': 'none', 'alpha': None, 'beta': None, 'mix_lambda': 0.0, 'start_epoch': None}
Epoch 001/10 | Train Loss 2.2094 | Train Acc 0.2317 | Train MacroF1 0.0858 | Val Loss 2.1228 | Val Acc 0.2510 | Val MacroF1 0.0916 | Val WeightedF1 0.1378 | Val Conf 0.2379 | Grad 1.5651 | FracApplied 0.0 | Throughput 527.2 samp/s | Time 7.59s
Epoch 002/10 | Train Loss 2.0316 | Train Acc 0.2907 | Train MacroF1 0.1842 | Val Loss 2.0557 | Val Acc 0.2870 | Val MacroF1 0.1951 | Val WeightedF1 0.2368 | Val Conf 0.2823 | Grad 1.5855 | FracApplied 0.0 | Throughput 543.1 samp/s | Time 7.36s
Epoch 003/10 | Train Loss 1.9588 | Train Acc 0.3190 | Train MacroF1 0.2325 | Val Loss 2.0059

/tmp/ipykernel_152190/478453138.py:531: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(


Epoch 001/10 | Train Loss 2.2341 | Train Acc 0.2165 | Train MacroF1 0.0750 | Val Loss 2.1354 | Val Acc 0.2330 | Val MacroF1 0.0761 | Val WeightedF1 0.1203 | Val Conf 0.2306 | Grad 1.5596 | FracApplied 0.0 | Throughput 541.0 samp/s | Time 7.39s
Epoch 002/10 | Train Loss 2.0606 | Train Acc 0.2743 | Train MacroF1 0.1617 | Val Loss 2.0045 | Val Acc 0.3090 | Val MacroF1 0.2046 | Val WeightedF1 0.2517 | Val Conf 0.2969 | Grad 1.5775 | FracApplied 0.0 | Throughput 499.5 samp/s | Time 8.01s
Epoch 003/10 | Train Loss 1.9666 | Train Acc 0.3110 | Train MacroF1 0.2200 | Val Loss 1.9638 | Val Acc 0.3170 | Val MacroF1 0.2435 | Val WeightedF1 0.2875 | Val Conf 0.3072 | Grad 1.6076 | FracApplied 0.0 | Throughput 494.2 samp/s | Time 8.09s
Epoch 004/10 | Train Loss 1.9320 | Train Acc 0.3295 | Train MacroF1 0.2513 | Val Loss 1.9481 | Val Acc 0.3110 | Val MacroF1 0.2327 | Val WeightedF1 0.2826 | Val Conf 0.3149 | Grad 1.6338 | FracApplied 0.0 | Throughput 532.5 samp/s | Time 7.51s
Epoch 005/10 | Train Los

...

Epoch 005/10 | Train Loss 2.0865 | Train Acc 0.2598 | Train MacroF1 0.1652 | Val Loss 2.1243 | Val Acc 0.2670 | Val MacroF1 0.1125 | Val WeightedF1 0.1632 | Val Conf 0.4104 | Grad 0.9273 | FracApplied 42.0 | Throughput 270.7 samp/s | Time 14.77s
Epoch 006/10 | Train Loss 2.0482 | Train Acc 0.2702 | Train MacroF1 0.1839 | Val Loss 2.0449 | Val Acc 0.2630 | Val MacroF1 0.1510 | Val WeightedF1 0.2105 | Val Conf 0.2657 | Grad 0.8629 | FracApplied 42.0 | Throughput 266.9 samp/s | Time 14.98s
Epoch 007/10 | Train Loss 1.9694 | Train Acc 0.3022 | Train MacroF1 0.2157 | Val Loss 1.9713 | Val Acc 0.2860 | Val MacroF1 0.1832 | Val WeightedF1 0.2450 | Val Conf 0.3211 | Grad 0.6040 | FracApplied 42.0 | Throughput 271.7 samp/s | Time 14.72s
Epoch 008/10 | Train Loss 1.9387 | Train Acc 0.3088 | Train MacroF1 0.2155 | Val Loss 1.9523 | Val Acc 0.3060 | Val MacroF1 0.2069 | Val WeightedF1 0.2703 | Val Conf 0.3096 | Grad 0.5509 | FracApplied 42.0 | Throughput 273.2 samp/s | Time 14.64s
Epoch 009/10 | T

Epoch 007/10 | Train Loss 1.9585 | Train Acc 0.3050 | Train MacroF1 0.2279 | Val Loss 2.0586 | Val Acc 0.2680 | Val MacroF1 0.1371 | Val WeightedF1 0.1999 | Val Conf 0.3347 | Grad 0.7699 | FracApplied 42.0 | Throughput 262.2 samp/s | Time 15.26s
Epoch 008/10 | Train Loss 1.9474 | Train Acc 0.3030 | Train MacroF1 0.2303 | Val Loss 2.0274 | Val Acc 0.2540 | Val MacroF1 0.1935 | Val WeightedF1 0.2591 | Val Conf 0.2556 | Grad 0.7707 | FracApplied 42.0 | Throughput 266.7 samp/s | Time 15.00s
Epoch 009/10 | Train Loss 1.9414 | Train Acc 0.3217 | Train MacroF1 0.2594 | Val Loss 2.0067 | Val Acc 0.2770 | Val MacroF1 0.1619 | Val WeightedF1 0.2260 | Val Conf 0.3890 | Grad 0.9625 | FracApplied 42.0 | Throughput 264.6 samp/s | Time 15.11s
Epoch 010/10 | Train Loss 1.9075 | Train Acc 0.3280 | Train MacroF1 0.2536 | Val Loss 1.9641 | Val Acc 0.3000 | Val MacroF1 0.2031 | Val WeightedF1 0.2663 | Val Conf 0.3644 | Grad 0.7823 | FracApplied 42.0 | Throughput 265.3 samp/s | Time 15.08s
----------------

Epoch 009/10 | Train Loss 1.6986 | Train Acc 0.3880 | Train MacroF1 0.3225 | Val Loss 1.8520 | Val Acc 0.3350 | Val MacroF1 0.2764 | Val WeightedF1 0.3198 | Val Conf 0.3773 | Grad 1.4785 | FracApplied 42.0 | Throughput 268.5 samp/s | Time 14.90s
Epoch 010/10 | Train Loss 1.6395 | Train Acc 0.4050 | Train MacroF1 0.3444 | Val Loss 1.8826 | Val Acc 0.3210 | Val MacroF1 0.2598 | Val WeightedF1 0.3009 | Val Conf 0.4293 | Grad 1.4934 | FracApplied 42.0 | Throughput 273.4 samp/s | Time 14.63s
----------------------------------------------------------------------------------------------------
BEST EPOCH          : 9
BEST VAL MACRO F1   : 0.276361
BEST VAL ACC        : 0.335000
BEST VAL LOSS       : 1.851985
TOTAL RUN TIME      : 150.79 sec
----------------------------------------------------------------------------------------------------

Summary updated:
  /home/tahiti/Malashin_Projects/summary_fractional_scenarios.json
  /home/tahiti/Malashin_Projects/summary_fractional_scenarios.csv

####

Epoch 001/10 | Train Loss 2.1908 | Train Acc 0.2257 | Train MacroF1 0.1036 | Val Loss 2.0763 | Val Acc 0.2450 | Val MacroF1 0.1075 | Val WeightedF1 0.1634 | Val Conf 0.2756 | Grad 1.0379 | FracApplied 42.0 | Throughput 260.9 samp/s | Time 15.33s
Epoch 002/10 | Train Loss 2.0473 | Train Acc 0.2625 | Train MacroF1 0.1678 | Val Loss 2.0014 | Val Acc 0.2950 | Val MacroF1 0.1856 | Val WeightedF1 0.2502 | Val Conf 0.2678 | Grad 1.0921 | FracApplied 42.0 | Throughput 257.7 samp/s | Time 15.52s
Epoch 003/10 | Train Loss 1.9789 | Train Acc 0.2990 | Train MacroF1 0.2175 | Val Loss 2.0008 | Val Acc 0.2910 | Val MacroF1 0.1656 | Val WeightedF1 0.2202 | Val Conf 0.3610 | Grad 1.0028 | FracApplied 42.0 | Throughput 251.0 samp/s | Time 15.93s
Epoch 004/10 | Train Loss 1.9207 | Train Acc 0.3110 | Train MacroF1 0.2342 | Val Loss 1.9494 | Val Acc 0.3070 | Val MacroF1 0.2261 | Val WeightedF1 0.2825 | Val Conf 0.3179 | Grad 1.0054 | FracApplied 42.0 | Throughput 264.1 samp/s | Time 15.15s
Epoch 005/10 | T


KeyboardInterrupt

